# 数理変換タクティクス AI トレーニング
## ハイブリッドEmbedding版（数学特徴 + 学習特徴 + 継続学習 + リソース対応）

In [ ]:
!pip install tensorflowjs scikit-learn optuna japanize-matplotlib -q
print('完了！')

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflowjs as tfjs
import random, os, math
import japanize_matplotlib
from sklearn.model_selection import train_test_split

MAX_HAND    = 20
EMBED_DIM   = 16
NUM_ACTIONS = 7
MAX_CARD    = 150
PRIMES      = [2, 3, 5, 7, 11, 13]
MATH_DIM    = len(PRIMES) + 3
SCALAR_DIM  = 10
DIM_NAMES   = ['2の指数','3の指数','5の指数','7の指数','11の指数','13の指数','大素数フラグ','約数の個数','桁の和']

print(f'TensorFlow: {tf.__version__}')

In [ ]:
# ================================================================
# ⚙️ 設定セル ── ここだけ変更すればOK！
# ================================================================

# ─── データ生成 ──────────────────────────────────────────────────
NUM_EPISODES = 30000     # 生成ゲーム数（多い→精度UP・時間もかかる）

# ─── ルール多様性 ────────────────────────────────────────────────
RULE_CONFIGS = [
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':5,   'resourceMin':0,  'resourceLogBase':2},
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':10,  'resourceMin':0,  'resourceLogBase':2},
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':10,  'resourceMin':5,  'resourceLogBase':2},
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':15,  'resourceMin':0,  'resourceLogBase':2},
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':15,  'resourceMin':10, 'resourceLogBase':2},
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':None,'resourceMin':0,  'resourceLogBase':2},
]

# ─── 学習設定 ─────────────────────────────────────────────────────
EPOCHS              = 60
BATCH_SIZE          = 256
LEARNING_RATE       = 0.001
EARLY_STOP_PATIENCE = 6
LR_REDUCE_PATIENCE  = 3

# ─── 教師学習のスキップ ───────────────────────────────────────────
# True: 保存済みモデルが読み込めた場合は教師学習(model.fit)を省略し、
#       RL微調整だけ行う（USE_RL=Falseなら何も学習せず終わるので注意）
SKIP_SUPERVISED_IF_MODEL_EXISTS = False

# ─── 数学特徴 重要度テスト ──────────────────────────────────────
IMPORTANCE_INTERVAL = 0      # 0=学習後のみ / 例:10=10エポックごと

# ─── Round 1 Optuna: ハイパーパラメータ＋特徴の初期探索 ─────────
USE_OPTUNA       = True
OPTUNA_TRIALS    = 20
OPTUNA_EPOCHS    = 15
OPTUNA_INTERVAL  = 5         # 何セッションに1回Optunaを実行するか（1=毎回）

# ─── Round 2 Optuna: 学習後の数学特徴入れ替え最適化 ─────────────
USE_ROUND2_OPTUNA      = False
ROUND2_TRIALS          = 30
ROUND2_FINETUNE_EPOCHS = 15
ROUND2_FINAL_EPOCHS    = 30
ROUND2_CORE_FEATURES   = [0, 1, 2, 7, 8]  # 0=2の指数 1=3の指数 2=5の指数 7=約数の個数 8=桁の和

# ─── 強化学習 (RL) ───────────────────────────────────────────────
# 教師学習で基礎を作った後にRL微調整でルールAIの限界を超えます
USE_RL             = False   # True にすると教師学習後にRL微調整を実行
RL_EPISODES        = 5000    # 対戦ゲーム数（多いほど強くなる）
RL_LEARNING_RATE   = 1e-4    # 学習率（教師学習より小さくする）
RL_GAMMA           = 0.99    # 割引率（1.0=将来重視、0.9=直近重視）
RL_ENTROPY_COEF    = 0.01    # 探索ボーナス（大きいほど多様な行動を試す）
RL_OPPONENT        = 'rule'  # 'rule'=ルールAIと対戦 / 'self'=自己対戦
RL_UPDATE_INTERVAL = 32      # 何エピソードごとにパラメータを更新するか

print('設定読み込み完了！')
print(f'  ゲーム数: {NUM_EPISODES}  ルール数: {len(RULE_CONFIGS)}')
print(f'  エポック: {EPOCHS}  バッチ: {BATCH_SIZE}  学習率: {LEARNING_RATE}')
print(f'  教師学習スキップ: {"有効（保存済みモデルがあれば省略）" if SKIP_SUPERVISED_IF_MODEL_EXISTS else "無効（毎回学習）"}')
print(f'  重要度テスト: {"学習後のみ" if IMPORTANCE_INTERVAL == 0 else f"{IMPORTANCE_INTERVAL}エポックごと"}')
print(f'  Round 1 Optuna: {"有効 (" + str(OPTUNA_TRIALS) + "試行, " + str(OPTUNA_INTERVAL) + "回に1回)" if USE_OPTUNA else "無効"}')
print(f'  Round 2 Optuna: {"有効 (" + str(ROUND2_TRIALS) + "試行)" if USE_ROUND2_OPTUNA else "無効"}')
print(f'  強化学習:       {"有効 (" + str(RL_EPISODES) + "ep, " + RL_OPPONENT + ")" if USE_RL else "無効"}')

## Google Drive 連携

In [ ]:
import shutil

def _detect_env():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        pass
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    return 'local'

ENV = _detect_env()

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/math_tactics_ai'
elif ENV == 'kaggle':
    SAVE_DIR = '/kaggle/working/math_tactics_ai'
    # 前回の出力を「Add Data」で入力データセットとして追加した場合のパス
    # （実際に付けたデータセット名に合わせて書き換えてください）
    KAGGLE_PREV_DIR = '/kaggle/input/math-tactics-ai'
    if os.path.exists(KAGGLE_PREV_DIR) and not os.path.exists(SAVE_DIR):
        shutil.copytree(KAGGLE_PREV_DIR, SAVE_DIR)
        print(f'前回の出力を引き継ぎました: {KAGGLE_PREV_DIR} → {SAVE_DIR}')
else:
    SAVE_DIR = './math_tactics_ai'

SAVE_PATH = os.path.join(SAVE_DIR, 'ai_best.keras')
DATA_PATH = os.path.join(SAVE_DIR, 'train_data_v4.npz')
SESSION_COUNT_PATH = os.path.join(SAVE_DIR, 'session_count.txt')
os.makedirs(SAVE_DIR, exist_ok=True)

# セッション回数を読み込み
session_count = 0
if os.path.exists(SESSION_COUNT_PATH):
    with open(SESSION_COUNT_PATH) as f:
        session_count = int(f.read().strip())

# このセッションでOptunaを実行するか判定
run_optuna_this_session = USE_OPTUNA and (session_count % OPTUNA_INTERVAL == 0)
next_optuna_in = OPTUNA_INTERVAL - (session_count % OPTUNA_INTERVAL)

print(f'実行環境: {ENV}')
print(f'モデル保存先: {SAVE_PATH}')
print(f'データ保存先: {DATA_PATH}')
print(f'セッション回数: {session_count + 1}回目')
if USE_OPTUNA:
    if run_optuna_this_session:
        print(f'Optuna: 今回実行（{OPTUNA_INTERVAL}回に1回）')
    else:
        print(f'Optuna: 今回スキップ（次回まであと {next_optuna_in} 回）')
if os.path.exists(SAVE_PATH): print('保存済みモデルあり')
if os.path.exists(DATA_PATH): print('保存済みデータあり')

## 数学特徴テーブル

In [ ]:
def _card_features_full(n):
    """カード値 n の全数学特徴を計算（14次元）"""
    if n == 0: return [0.0] * 14
    orig = n
    # ── 1〜6: 素因数の指数 (2,3,5,7,11,13) ──
    feats = []
    tmp = n
    for p in PRIMES:
        c = 0
        while tmp % p == 0: tmp //= p; c += 1
        feats.append(min(c, 4) / 4.0)
    # ── 7: 大素数フラグ (13より大きい素因数あり) ──
    feats.append(1.0 if tmp > 1 else 0.0)
    # ── 8: 約数の個数 ──
    feats.append(min(sum(1 for d in range(1, orig+1) if orig % d == 0), 20) / 20.0)
    # ── 9: 桁の和 ──
    feats.append(min(sum(int(d) for d in str(orig)), 30) / 30.0)
    # ── 10: 素数フラグ ──
    is_p = orig > 1 and all(orig % d != 0 for d in range(2, int(orig**0.5)+1))
    feats.append(1.0 if is_p else 0.0)
    # ── 11: 最小素因数（正規化） ──
    min_pf = orig
    for d in range(2, int(orig**0.5)+1):
        if orig % d == 0: min_pf = d; break
    feats.append(min(min_pf, 150) / 150.0)
    # ── 12: 最大素因数（正規化） ──
    max_pf, tmp2 = 1, orig
    for d in range(2, int(orig**0.5)+1):
        while tmp2 % d == 0: tmp2 //= d; max_pf = d
    if tmp2 > 1: max_pf = tmp2
    feats.append(min(max_pf, 150) / 150.0)
    # ── 13: 素因数の個数Ω（重複あり） ──
    omega, tmp3 = 0, orig
    for p in list(PRIMES) + list(range(17, int(orig**0.5)+1)):
        while tmp3 % p == 0: tmp3 //= p; omega += 1
    if tmp3 > 1: omega += 1
    feats.append(min(omega, 8) / 8.0)
    # ── 14: 約数の和 σ(n)（正規化） ──
    sigma = sum(d for d in range(1, orig+1) if orig % d == 0)
    feats.append(min(sigma, orig * 6) / max(orig * 6, 1))
    return feats

# 全特徴テーブル（151枚 × 14次元）
ALL_DIM_NAMES = [
    '2の指数','3の指数','5の指数','7の指数','11の指数','13の指数',  # 0-5
    '大素数フラグ','約数の個数','桁の和',                           # 6-8（ここまで従来）
    '素数フラグ','最小素因数','最大素因数','素因数個数Ω','約数の和σ', # 9-13（新規）
]
ALL_FACTOR_TABLE = np.array([_card_features_full(i) for i in range(MAX_CARD+1)], dtype=np.float32)

# デフォルト: 従来の9次元（Optunaが最適化するまでの初期値）
MATH_FEATURE_MASK = list(range(9))
FACTOR_TABLE = ALL_FACTOR_TABLE[:, MATH_FEATURE_MASK]
MATH_DIM     = len(MATH_FEATURE_MASK)
DIM_NAMES    = [ALL_DIM_NAMES[i] for i in MATH_FEATURE_MASK]

print(f'全特徴テーブル: {ALL_FACTOR_TABLE.shape}')
print(f'使用中: {MATH_DIM}次元 → {DIM_NAMES}')

## ゲームシミュレーター（リソース対応）

In [ ]:
    def reset(self):
        cfg=self.config
        self.hands={'P1':[random.randint(1,cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],'P2':[random.randint(1,cfg['initMaxNum']) for _ in range(cfg['initHandCount'])]}
        self.scores={'P1':0,'P2':0}; self.turn_count=1; self.current_player='P1'
        self.done=False; self.winner=None
        ri = cfg.get('resourceInitial', None)
        self.resources = {'P1': ri, 'P2': ri}
    def draw(self,pid):
        """ターン開始時のドロー（最初のラウンドはP1・P2ともドローなし）"""
        if self.turn_count==1: return
        cfg=self.config
        for _ in range(cfg.get('drawCount',0)):
            self.hands[pid].append(random.randint(1,cfg.get('drawMaxNum',10)))
    def opponent(self,pid): return 'P2' if pid=='P1' else 'P1'

## データ生成

## モデル作成・読込

In [ ]:
if os.path.exists(DATA_PATH):
    print('保存済みデータを読み込みます: ' + DATA_PATH)
    d=np.load(DATA_PATH)
    my_hands_data=d['my_hands']; enemy_hands_data=d['enemy_hands']
    scalars_data=d['scalars'];   y_data=d['y']
    print('読み込み完了！ %d 件' % len(y_data))
else:
    print('データ生成を開始します... (%d ゲーム × %d ルール)' % (NUM_EPISODES, len(RULE_CONFIGS)))
    # Pythonリストの代わりに最初からnumpy配列を確保
    # （Pythonリストは1件あたり約1000バイト → 数百万件で数GB消費するため）
    ESTIMATED = NUM_EPISODES * 200
    my_hands_data    = np.zeros((ESTIMATED, MAX_HAND),   dtype=np.int32)
    enemy_hands_data = np.zeros((ESTIMATED, MAX_HAND),   dtype=np.int32)
    scalars_data     = np.zeros((ESTIMATED, SCALAR_DIM), dtype=np.float32)
    y_data           = np.zeros(ESTIMATED,               dtype=np.int32)
    n = 0

    for ep in range(NUM_EPISODES):
        cfg=random.choice(RULE_CONFIGS); game=MathCardGame(cfg); game.reset()
        while not game.done:
            pid=game.current_player
            game.draw(pid)
            while not game.done:
                my_h,enm_h,sc=game.get_inputs(pid)
                action=game.get_optimal_action(pid)
                my_hands_data[n]=my_h; enemy_hands_data[n]=enm_h
                scalars_data[n]=sc;    y_data[n]=action
                n += 1
                if action==0:
                    game.switch_player()
                    break
                game.execute_action_inplace(pid,action)
        if (ep+1)%10000==0: print('%d/%d  サンプル数: %d' % (ep+1,NUM_EPISODES,n))

    my_hands_data    = my_hands_data[:n]
    enemy_hands_data = enemy_hands_data[:n]
    scalars_data     = scalars_data[:n]
    y_data           = y_data[:n]
    print('生成完了！ 総データ数: %d' % n)
    np.savez_compressed(DATA_PATH,my_hands=my_hands_data,enemy_hands=enemy_hands_data,scalars=scalars_data,y=y_data)
    print('データを保存しました: ' + DATA_PATH)

In [ ]:
from tensorflow.keras.utils import to_categorical

idx=np.arange(len(y_data))
train_idx,val_idx=train_test_split(idx,test_size=0.1,random_state=42)

def make_dataset(indices, shuffle=True, bs=None):
    ds=tf.data.Dataset.from_tensor_slices((
        {'my_hand':my_hands_data[indices],'enemy_hand':enemy_hands_data[indices],'scalars':scalars_data[indices]},
        to_categorical(y_data[indices],NUM_ACTIONS)))
    if shuffle: ds=ds.shuffle(10000)
    return ds.batch(bs or BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_dataset(train_idx); val_ds=make_dataset(val_idx,shuffle=False)

# Optunaでアーキテクチャが変わった場合は旧モデルを破棄する
if run_optuna_this_session and os.path.exists(SAVE_PATH):
    _chk = tf.keras.models.load_model(SAVE_PATH)
    _old_math = _chk.get_layer('math_emb').get_weights()[0].shape[1]
    _old_emb  = _chk.get_layer('learned_emb').get_weights()[0].shape[1]
    del _chk
    if _old_math != MATH_DIM or _old_emb != EMBED_DIM:
        os.remove(SAVE_PATH)
        print(f'アーキテクチャ変更検出: MATH_DIM {_old_math}→{MATH_DIM}, EMBED_DIM {_old_emb}→{EMBED_DIM}')
        print('旧モデルを削除して新規作成します')

if os.path.exists(SAVE_PATH):
    print('保存済みモデルを読み込み中...')
    model=tf.keras.models.load_model(SAVE_PATH)
    model.get_layer('math_emb').set_weights([FACTOR_TABLE])
    model_loaded_from_disk = True
    print('読み込み完了！続きから学習します。')
else:
    print('新規モデルを作成します...')
    my_hand_in   =tf.keras.Input(shape=(MAX_HAND,),  dtype='int32',  name='my_hand')
    enemy_hand_in=tf.keras.Input(shape=(MAX_HAND,),  dtype='int32',  name='enemy_hand')
    scalar_in    =tf.keras.Input(shape=(SCALAR_DIM,),dtype='float32',name='scalars')
    math_emb   =tf.keras.layers.Embedding(MAX_CARD+1,MATH_DIM,weights=[FACTOR_TABLE],trainable=False,mask_zero=True,name='math_emb')
    learned_emb=tf.keras.layers.Embedding(MAX_CARD+1,EMBED_DIM,mask_zero=True,name='learned_emb')
    def encode_hand(hand_input):
        combined=tf.keras.layers.Concatenate(axis=-1)([math_emb(hand_input),learned_emb(hand_input)])
        return tf.keras.layers.Flatten()(combined)
    x=tf.keras.layers.Concatenate(name='combined')([encode_hand(my_hand_in),encode_hand(enemy_hand_in),scalar_in])
    x=tf.keras.layers.Dense(256,activation='relu')(x)
    x=tf.keras.layers.Dropout(0.3)(x)
    x=tf.keras.layers.Dense(128,activation='relu')(x)
    x=tf.keras.layers.Dropout(0.2)(x)
    out=tf.keras.layers.Dense(NUM_ACTIONS,activation='softmax',name='action')(x)
    model=tf.keras.Model(inputs=[my_hand_in,enemy_hand_in,scalar_in],outputs=out)
    model_loaded_from_disk = False
    print('新規モデル作成完了！')

model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),loss='categorical_crossentropy',metrics=['accuracy'])
total=model.count_params(); trainable=sum(tf.size(w).numpy() for w in model.trainable_weights)
print('総パラメータ: %d  学習: %d  固定: %d' % (total,trainable,total-trainable))

## 学習メイン

## Optuna ハイパーパラメータ自動探索

In [ ]:
import optuna, gc
optuna.logging.set_verbosity(optuna.logging.WARNING)

# インデックス0〜8は固定、9以降は ALL_DIM_NAMES から自動生成
# → 新しい数学特徴を cell-7 に追加するだけで自動的に探索対象になる
_OPTIONAL_FEATS = {i: ALL_DIM_NAMES[i] for i in range(9, len(ALL_DIM_NAMES))}

def _build_trial_model(trial):
    """Optuna trial 用モデルビルダー（特徴選択 + ハイパーパラメータ探索）"""
    # ── 特徴選択 ──
    mask = list(range(9))  # 0〜8は固定
    for idx, name in _OPTIONAL_FEATS.items():
        if trial.suggest_categorical(f'feat_{name}', [True, False]):
            mask.append(idx)
    ft       = ALL_FACTOR_TABLE[:, mask]
    math_dim = len(mask)

    # ── ハイパーパラメータ ──
    lr      = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    bs      = trial.suggest_categorical('batch_size', [128, 256, 512])
    units1  = trial.suggest_categorical('units1', [128, 256, 512])
    units2  = trial.suggest_categorical('units2', [64, 128, 256])
    drop1   = trial.suggest_float('dropout1', 0.1, 0.5)
    drop2   = trial.suggest_float('dropout2', 0.1, 0.4)
    emb_dim = trial.suggest_categorical('embed_dim', [8, 16, 32])

    my_in  = tf.keras.Input(shape=(MAX_HAND,),   dtype='int32',   name='my_hand')
    enm_in = tf.keras.Input(shape=(MAX_HAND,),   dtype='int32',   name='enemy_hand')
    sc_in  = tf.keras.Input(shape=(SCALAR_DIM,), dtype='float32', name='scalars')
    me = tf.keras.layers.Embedding(MAX_CARD+1, math_dim, weights=[ft],  trainable=False, mask_zero=True, name='math_emb')
    le = tf.keras.layers.Embedding(MAX_CARD+1, emb_dim,                 mask_zero=True,                  name='learned_emb')
    def enc(h): return tf.keras.layers.Flatten()(tf.keras.layers.Concatenate(axis=-1)([me(h), le(h)]))
    x = tf.keras.layers.Concatenate(name='combined')([enc(my_in), enc(enm_in), sc_in])
    x = tf.keras.layers.Dense(units1, activation='relu')(x)
    x = tf.keras.layers.Dropout(drop1)(x)
    x = tf.keras.layers.Dense(units2, activation='relu')(x)
    x = tf.keras.layers.Dropout(drop2)(x)
    out = tf.keras.layers.Dense(NUM_ACTIONS, activation='softmax', name='action')(x)
    m = tf.keras.Model(inputs=[my_in, enm_in, sc_in], outputs=out)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='categorical_crossentropy', metrics=['accuracy'])
    return m, bs, mask

def _objective(trial):
    tf.keras.backend.clear_session()   # 前のトライアルのグラフをクリア
    m, bs, _ = _build_trial_model(trial)
    tr = make_dataset(train_idx, bs=bs)
    va = make_dataset(val_idx,   bs=bs, shuffle=False)
    h = m.fit(tr, validation_data=va, epochs=OPTUNA_EPOCHS,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)],
              verbose=0)
    val_acc = max(h.history['val_accuracy'])
    del m, tr, va, h   # メモリ解放
    gc.collect()
    return val_acc

if run_optuna_this_session:
    print(f'Optuna 探索開始: {OPTUNA_TRIALS}試行 × {OPTUNA_EPOCHS}エポック')
    print(f'探索対象: 特徴選択({len(_OPTIONAL_FEATS)}種) + lr / batch / embed / units / dropout')
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

    best = study.best_params
    print(f'\n✅ 最適パラメータ (val_accuracy={study.best_value:.4f}):')

    best_mask = list(range(9))
    for idx, name in _OPTIONAL_FEATS.items():
        use = best.get(f'feat_{name}', False)
        mark = '✅' if use else '❌'
        print(f'  {mark} {name}')
        if use: best_mask.append(idx)
    print(f'  learning_rate: {best["learning_rate"]:.5f}')
    print(f'  batch_size:    {best["batch_size"]}')
    print(f'  embed_dim:     {best["embed_dim"]}')
    print(f'  units1/2:      {best["units1"]} / {best["units2"]}')

    # Optunaが終わったらメインモデル用にセッションを再度クリア
    tf.keras.backend.clear_session()
    gc.collect()

    MATH_FEATURE_MASK = best_mask
    FACTOR_TABLE      = ALL_FACTOR_TABLE[:, MATH_FEATURE_MASK]
    MATH_DIM          = len(MATH_FEATURE_MASK)
    DIM_NAMES         = [ALL_DIM_NAMES[i] for i in MATH_FEATURE_MASK]
    LEARNING_RATE     = best['learning_rate']
    BATCH_SIZE        = best['batch_size']
    EMBED_DIM         = best['embed_dim']
    train_ds = make_dataset(train_idx, bs=BATCH_SIZE)
    val_ds   = make_dataset(val_idx,   bs=BATCH_SIZE, shuffle=False)
    print(f'\n→ 特徴次元: {MATH_DIM}  EMBED_DIM: {EMBED_DIM}')
else:
    if USE_OPTUNA:
        print(f'Optuna スキップ（{next_optuna_in}回後に実行予定）')
    else:
        print('Optuna 無効 (USE_OPTUNA=False)')

In [ ]:
class ImportanceTestCallback(tf.keras.callbacks.Callback):
    """IMPORTANCE_INTERVAL エポックごとに数学特徴の重要度テストを実行"""
    def on_epoch_end(self, epoch, logs=None):
        if IMPORTANCE_INTERVAL <= 0: return
        if (epoch + 1) % IMPORTANCE_INTERVAL != 0: return
        print(f'\n─── 重要度テスト (epoch {epoch+1}) ───')
        _, base_acc = self.model.evaluate(val_ds, verbose=0)
        for d in range(MATH_DIM):
            shuffled = FACTOR_TABLE.copy()
            shuffled[:,d] = np.random.permutation(shuffled[:,d])
            self.model.get_layer('math_emb').set_weights([shuffled])
            _, acc = self.model.evaluate(val_ds, verbose=0)
            drop = base_acc - acc
            self.model.get_layer('math_emb').set_weights([FACTOR_TABLE])
            bar = '█' * max(0, int(drop * 200)) + ('▒' if drop <= 0 else '')
            print(f'  {DIM_NAMES[d]:<12}: {drop*100:+.2f}%  {bar}')

if SKIP_SUPERVISED_IF_MODEL_EXISTS and model_loaded_from_disk:
    print('教師学習をスキップしました（SKIP_SUPERVISED_IF_MODEL_EXISTS=True、保存済みモデルを使用）')
    history = None
else:
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=EARLY_STOP_PATIENCE, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=LR_REDUCE_PATIENCE),
        tf.keras.callbacks.ModelCheckpoint(filepath=SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
        ImportanceTestCallback(),
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)
    best_val = max(history.history['val_accuracy'])
    print('最高検証精度: %.1f%%' % (best_val * 100))

# セッションカウントを更新して保存
session_count += 1
with open(SESSION_COUNT_PATH, 'w') as f:
    f.write(str(session_count))
next_optuna_in = OPTUNA_INTERVAL - (session_count % OPTUNA_INTERVAL)
print(f'セッション {session_count} 完了！次のOptuna: あと {next_optuna_in} 回')

In [ ]:
# ============================================================
# 🔄 Round 2 Optuna: 数学特徴入れ替え最適化
# ============================================================
# 学習済みモデルの構造を変えずに math_emb だけ差し替えて
# より良い特徴の組み合わせを探索します。
# learned_emb（ゲーム経験の重み）はそのまま保持されます。
#
# 使い方:
#   1. 上の学習セルを実行してモデルを保存する
#   2. 設定セルで USE_ROUND2_OPTUNA = True にする
#   3. このセルを実行する

if USE_ROUND2_OPTUNA:
    if not os.path.exists(SAVE_PATH):
        print('⚠️  保存済みモデルが見つかりません。先に学習セルを実行してください。')
    else:
        # 保存済みモデルの MATH_DIM を読み取る（構造を壊さないため固定）
        _r2_base = tf.keras.models.load_model(SAVE_PATH)
        _r2_math_dim = _r2_base.get_layer('math_emb').get_weights()[0].shape[1]
        del _r2_base

        _r2_non_core    = [i for i in range(len(ALL_DIM_NAMES)) if i not in ROUND2_CORE_FEATURES]
        _r2_extra_slots = _r2_math_dim - len(ROUND2_CORE_FEATURES)

        print('Round 2 Optuna 開始')
        print(f'  保存済みモデルの MATH_DIM = {_r2_math_dim}（固定・アーキテクチャ維持）')
        print(f'  コア特徴 ({len(ROUND2_CORE_FEATURES)}個): {[ALL_DIM_NAMES[i] for i in ROUND2_CORE_FEATURES]}')
        print(f'  探索スロット: {_r2_extra_slots}個  候補: {len(_r2_non_core)}個')
        print(f'  {ROUND2_TRIALS}試行 × {ROUND2_FINETUNE_EPOCHS}エポック / 試行\n')

        if _r2_extra_slots < 0:
            print(f'⚠️  ROUND2_CORE_FEATURES ({len(ROUND2_CORE_FEATURES)}個) が')
            print(f'   保存済みモデルの MATH_DIM ({_r2_math_dim}) を超えています。')
            print('   設定セルの ROUND2_CORE_FEATURES を減らしてください。')
        else:
            def _r2_mask_from_params(params):
                """優先度スコアから MATH_DIM 個の特徴マスクを生成"""
                scored = sorted(_r2_non_core, key=lambda i: -params.get(f'prio_{i}', 0.0))
                return sorted(ROUND2_CORE_FEATURES + scored[:_r2_extra_slots])

            def _r2_objective(trial):
                # 非コア特徴に優先度スコアを付与（連続値でTPEが探索しやすい）
                params = {f'prio_{i}': trial.suggest_float(f'prio_{i}', 0.0, 1.0)
                          for i in _r2_non_core}
                mask = _r2_mask_from_params(params)

                m = tf.keras.models.load_model(SAVE_PATH)
                m.get_layer('math_emb').set_weights([ALL_FACTOR_TABLE[:, mask]])
                m.get_layer('learned_emb').trainable = False  # 経験を凍結
                m.compile(
                    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE * 0.1),
                    loss='categorical_crossentropy', metrics=['accuracy']
                )
                h = m.fit(
                    make_dataset(train_idx, bs=BATCH_SIZE),
                    validation_data=make_dataset(val_idx, shuffle=False, bs=BATCH_SIZE),
                    epochs=ROUND2_FINETUNE_EPOCHS,
                    callbacks=[tf.keras.callbacks.EarlyStopping(
                        monitor='val_accuracy', patience=3, restore_best_weights=True)],
                    verbose=0
                )
                return max(h.history.get('val_accuracy', [0.0]))

            def _r2_callback(study, trial):
                mask  = _r2_mask_from_params(trial.params)
                names = [ALL_DIM_NAMES[i] for i in mask]
                print(f'  trial {trial.number:02d}: {trial.value:.4f}  → {names}')

            study2 = optuna.create_study(
                direction='maximize',
                sampler=optuna.samplers.TPESampler(seed=0)
            )
            study2.optimize(_r2_objective, n_trials=ROUND2_TRIALS, callbacks=[_r2_callback])

            # ── 最良特徴を確定してグローバル変数に反映 ──
            best2             = study2.best_trial
            MATH_FEATURE_MASK = _r2_mask_from_params(best2.params)
            FACTOR_TABLE      = ALL_FACTOR_TABLE[:, MATH_FEATURE_MASK]
            MATH_DIM          = len(MATH_FEATURE_MASK)
            DIM_NAMES         = [ALL_DIM_NAMES[i] for i in MATH_FEATURE_MASK]

            print(f'\n✅ Round 2 最良特徴 (val_accuracy={best2.value:.4f}):')
            for i in range(len(ALL_DIM_NAMES)):
                mark = '✅' if i in MATH_FEATURE_MASK else '❌'
                print(f'  {mark} [{i:02d}] {ALL_DIM_NAMES[i]}')

            # ── Phase 1: math_emb 差し替え + Dense層ファインチューニング ──
            print(f'\nPhase 1: Dense層ファインチューニング ({ROUND2_FINAL_EPOCHS}エポック)...')
            model = tf.keras.models.load_model(SAVE_PATH)
            model.get_layer('math_emb').set_weights([FACTOR_TABLE])
            model.get_layer('learned_emb').trainable = False
            model.compile(
                optimizer=tf.keras.optimizers.Adam(LEARNING_RATE * 0.1),
                loss='categorical_crossentropy', metrics=['accuracy']
            )
            model.fit(
                make_dataset(train_idx, bs=BATCH_SIZE),
                validation_data=make_dataset(val_idx, shuffle=False, bs=BATCH_SIZE),
                epochs=ROUND2_FINAL_EPOCHS,
                callbacks=[
                    tf.keras.callbacks.EarlyStopping(
                        monitor='val_accuracy', patience=EARLY_STOP_PATIENCE,
                        restore_best_weights=True),
                    tf.keras.callbacks.ModelCheckpoint(
                        filepath=SAVE_PATH, monitor='val_accuracy',
                        save_best_only=True, verbose=1)
                ]
            )

            # ── Phase 2: 全層解凍して微調整（非常に小さいLR） ──
            print('\nPhase 2: 全層解凍・最終微調整 (10エポック)...')
            model.get_layer('learned_emb').trainable = True
            model.compile(
                optimizer=tf.keras.optimizers.Adam(LEARNING_RATE * 0.01),
                loss='categorical_crossentropy', metrics=['accuracy']
            )
            model.fit(
                make_dataset(train_idx, bs=BATCH_SIZE),
                validation_data=make_dataset(val_idx, shuffle=False, bs=BATCH_SIZE),
                epochs=10,
                callbacks=[
                    tf.keras.callbacks.EarlyStopping(
                        monitor='val_accuracy', patience=3,
                        restore_best_weights=True),
                    tf.keras.callbacks.ModelCheckpoint(
                        filepath=SAVE_PATH, monitor='val_accuracy',
                        save_best_only=True, verbose=1)
                ]
            )
            print('\n🎉 Round 2 完了！モデルを保存しました。')
else:
    print('Round 2 Optuna スキップ (USE_ROUND2_OPTUNA=False)')
    print('設定セルで USE_ROUND2_OPTUNA=True にして再実行できます。')

In [ ]:
        for ep in range(1, RL_EPISODES + 1):
            cfg  = random.choice(RULE_CONFIGS)
            game = MathCardGame(cfg)
            game.reset()
            traj = {'P1': [], 'P2': []}

            while not game.done:
                pid = game.current_player
                game.draw(pid)
                # 1ターン：パスするまで同じプレイヤーが複数回アクション
                while not game.done:
                    my_h, enm_h, sc = game.get_inputs(pid)
                    if RL_OPPONENT == 'rule' and pid == 'P2':
                        action = game.get_optimal_action(pid)
                    else:
                        action = model_sample_action(my_h, enm_h, sc)
                    traj[pid].append((my_h, enm_h, sc, action))
                    if action == 0:        # パス → プレイヤー交代
                        game.switch_player()
                        break
                    game.execute_action_inplace(pid, action)

## 精度グラフ

In [ ]:
import matplotlib.pyplot as plt
if history is None:
    print('教師学習をスキップしたため精度グラフはありません。')
else:
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for ax,key,title in zip(axes,['accuracy','loss'],['精度','損失']):
        ax.plot(history.history[key],label='訓練')
        ax.plot(history.history['val_'+key],label='検証')
        ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()

## 数学特徴の重要度

In [ ]:
import matplotlib.pyplot as plt
base_loss,base_acc=model.evaluate(val_ds,verbose=0)
print('ベースライン精度: %.2f%%' % (base_acc*100))
importances=[]
for d in range(MATH_DIM):
    shuffled=FACTOR_TABLE.copy()
    shuffled[:,d]=np.random.permutation(shuffled[:,d])
    model.get_layer('math_emb').set_weights([shuffled])
    _,acc=model.evaluate(val_ds,verbose=0)
    drop=base_acc-acc; importances.append(drop)
    model.get_layer('math_emb').set_weights([FACTOR_TABLE])
    print('次元%2d [%s]: %+.2f%%' % (d+1,DIM_NAMES[d],drop*100))
plt.figure(figsize=(10,4))
colors=['crimson' if v>0.02 else 'steelblue' for v in importances]
plt.barh(DIM_NAMES,[v*100 for v in importances],color=colors)
plt.axvline(0,color='black',linewidth=0.8)
plt.xlabel('精度の低下 (%)')
plt.title('数学特徴の重要度')
plt.tight_layout(); plt.show()
print('最も重要: ' + DIM_NAMES[int(np.argmax(importances))])

## TensorFlow.js エクスポート

In [ ]:
import shutil, gc

# 学習データをメモリから解放してからエクスポート
try:
    del my_hands_data, enemy_hands_data, scalars_data, y_data, train_ds, val_ds
    gc.collect()
    print('学習データをメモリから解放しました')
except:
    pass

OUT='tfjs_model'
tfjs.converters.save_keras_model(model,OUT)
print('エクスポート完了！')
for f in os.listdir(OUT): print('  %s (%.1f KB)' % (f,os.path.getsize(os.path.join(OUT,f))/1024))
shutil.make_archive('tfjs_model','zip','tfjs_model')

if ENV == 'colab':
    from google.colab import files
    files.download('tfjs_model.zip')
    print('ダウンロード開始！')
else:
    out_zip = os.path.join(os.getcwd(), 'tfjs_model.zip')
    print(f'tfjs_model.zip を作成しました: {out_zip}')
    print('Kaggleの場合は画面右側の「Output」タブからダウンロードできます。')

## 動作テスト

In [ ]:
names=['パス','攻撃','足し算','引き算','商x余','桁和分裂','GCD掛け']
std_cfg={'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,
         'handLimitNum':MAX_CARD,'winScore':100,'maxTurns':10,'resourceInitial':10,'resourceLogBase':2}
tests=[
    {'p1':[6,3,10,7,2], 'p2':[12,18,5,9,4],    'desc':'攻撃チャンスあり'},
    {'p1':[5,8,3,2,7],  'p2':[1,1,1,1,1],       'desc':'合成が有利'},
    {'p1':[6,9,4,3,2],  'p2':[8,10,6,4,3],      'desc':'GCD有効'},
    {'p1':[36,12,24,48],'p2':[7,11,13,17,19,23],'desc':'相手が全員素数'}
]
for t in tests:
    g=MathCardGame(std_cfg); g.reset()
    g.hands['P1']=t['p1']; g.hands['P2']=t['p2']
    g.turn_count=3; g.scores={'P1':20,'P2':30}; g.resources={'P1':8.0,'P2':8.0}
    my_h,enm_h,sc=g.get_inputs('P1')
    pred=model.predict({'my_hand':np.array([my_h]),'enemy_hand':np.array([enm_h]),'scalars':np.array([sc])},verbose=0)[0]
    chosen=int(np.argmax(pred)); optimal=g.get_optimal_action('P1')
    mark='OK' if chosen==optimal else 'NG'
    print('[%s] %s  AI:%s  最適:%s' % (mark,t['desc'],names[chosen],names[optimal]))